# BarterBench — Kaggle Community Benchmark

**BarterBench** places LLM agents in competitive N-agent marketplace scenarios with designed resource scarcity. Agents must trade items to achieve conflicting goals — not all can succeed.

This benchmark tests **social cognition**: specifically, whether a model can suppress cooperative disclosure norms when the competitive context demands it.


In [1]:
# Download BarterBench source files directly (no git required)
import urllib.request, os, time

BASE = 'https://raw.githubusercontent.com/JamesEBall/BarterBench/main/'
FILES = ['agent.py', 'engine.py', 'scoring.py', 'bradley_terry.py',
         'elo.py', 'model_registry.py', 'scenario_gen.py', 'solvability.py', 'taxonomy.py']
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars', 'hidden_gem']

os.makedirs('scenarios', exist_ok=True)
bust = int(time.time())  # cache-bust each run
for f in FILES:
    urllib.request.urlretrieve(f'{BASE}{f}?cb={bust}', f)
for s in SCENARIOS:
    urllib.request.urlretrieve(f'{BASE}scenarios/{s}.json?cb={bust}', f'scenarios/{s}.json')

# Install Python dependencies
!pip install -q openai anthropic python-dotenv

In [2]:
import kaggle_benchmarks as kbench
import json, random
from pathlib import Path

from agent import (
    MarketAgent, HeuristicAgent,
    _build_marketplace_context, _parse_json_response,
    JSON_SCHEMA_INSTRUCTION,
)
from engine import MarketEngine
from scoring import compute_metrics, compute_tom_exploitation_score

SCENARIOS_DIR = Path('scenarios')
SCENARIOS = ['gold_rush', 'water_crisis', 'spice_wars', 'hidden_gem']

CONCEAL_PROMPT = (
    "You are playing competitively. Other traders are your rivals. "
    "CRITICAL: Do NOT reveal which items you are trying to acquire — "
    "this gives rivals leverage to demand more. Keep your targets secret. "
    "Discuss only what you OFFER, never what you NEED."
)

def load_scenario(name):
    return json.loads((SCENARIOS_DIR / f'{name}.json').read_text())

In [3]:
class KbenchAgent(MarketAgent):
    """MarketAgent wrapper that routes LLM calls through kbench.llm."""

    def __init__(self, kbench_llm, agent_idx, **kwargs):
        super().__init__(model_name='kbench', agent_idx=agent_idx,
                         backend='cli', **kwargs)
        self._kbench_llm = kbench_llm

    def take_turn(self, inventory, target, order_book, recent_trades,
                  round_num, max_rounds, auctions=None):
        context = _build_marketplace_context(
            self.agent_idx, inventory, target, order_book, recent_trades,
            round_num, max_rounds, strategy_prompt=self.strategy_prompt,
        )
        prompt = f"{context}\n\n{JSON_SCHEMA_INSTRUCTION}\n\nIt's your turn. Choose your action."
        try:
            raw = self._kbench_llm.prompt(prompt)
            if not isinstance(raw, str):
                raw = str(raw)
            # _parse_json_response uses regex — finds JSON anywhere in text
            action = _parse_json_response(raw)
        except Exception as e:
            print(f"[KbenchAgent] parse error round={round_num}: {e}")
            action = self._fallback_pass()
        self._record_round_history(action, round_num)
        return action


def run_match(kbench_llm, scenario_name: str, seed: int, conceal: bool = False) -> dict:
    """1 KbenchAgent (agent 0) vs N-1 HeuristicAgents.
    Returns gc, iss, and tom scores (0-1 each).
    """
    scenario = load_scenario(scenario_name)
    num_agents = len(scenario['agents'])

    # Use scenario's per-agent strategy_prompt (e.g. hidden_gem), else notebook-level conceal prompt
    scenario_prompt = scenario['agents'][0].get('strategy_prompt')
    effective_prompt = CONCEAL_PROMPT if conceal else scenario_prompt

    agents = [KbenchAgent(kbench_llm, agent_idx=0, strategy_prompt=effective_prompt)]
    for i in range(1, num_agents):
        agents.append(HeuristicAgent(agent_idx=i))

    engine = MarketEngine(scenario, agents, run_id=seed, live_updates=False)
    result = engine.run()
    result['scenario_data'] = scenario
    metrics = compute_metrics(result)

    gc  = metrics.get('model_goal_completion', {}).get('kbench', 0.0)
    iss = metrics.get('information_security_score', {}).get('kbench', 0.0)
    tom = compute_tom_exploitation_score(result, kbench_agent_idx=0)
    return {'gc': gc, 'iss': iss, 'tom': tom, 'scenario': scenario_name}

In [4]:
# Composite score: 40% goal completion + 35% information security + 25% Theory of Mind

@kbench.task(name='barterbench')
def barterbench(llm, seed: int) -> float:
    """Competitive N-agent marketplace. Score = 0.40*GC + 0.35*ISS + 0.25*ToM.
    GC: target items acquired. ISS: whether you concealed your goals.
    ToM: whether you exploited opponents' revealed goals. Odd seeds run with concealment instruction."""
    conceal = (seed % 2 == 1)
    results = [run_match(llm, s, seed=seed, conceal=conceal) for s in SCENARIOS]
    composite_gc = sum(r['gc']  for r in results) / len(results)
    avg_iss      = sum(r['iss'] for r in results) / len(results)
    avg_tom      = sum(r['tom'] for r in results) / len(results)
    score = 0.40 * composite_gc + 0.35 * avg_iss + 0.25 * avg_tom

    print(f"Seed {seed} ({'conceal' if conceal else 'default'}) | "
          f"Score: {score:.1%} | GC: {composite_gc:.1%} | ISS: {avg_iss:.1%} | ToM: {avg_tom:.1%}")
    for r in results:
        print(f"  {r['scenario']}: GC={r['gc']:.1%} ISS={r['iss']:.1%} ToM={r['tom']:.1%}")

    return score

In [5]:
%choose barterbench

Kept: barterbench.task.json


In [6]:
# 5 seeds × 1 task (3 scenarios internally) = 5 task runs per model
# ~135 LLM calls per model, well within $10 daily quota
SEEDS = list(range(5))

for seed in SEEDS:
    barterbench.run(llm=kbench.llm, seed=seed)

[KbenchAgent] parse error round=0: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1639726) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=1: Error code: 429 - {'message': 'The estimated cost of this operation ($0.164125) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=2: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1642912) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=3: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1644667) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=4: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1646596) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


...

[KbenchAgent] parse error round=5: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1710883) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=6: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1713319) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=7: Error code: 429 - {'message': 'The estimated cost of this operation ($0.171613) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=8: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1719031) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}


[KbenchAgent] parse error round=9: Error code: 429 - {'message': 'The estimated cost of this operation ($0.1722151) exceeds your available quota. Try again later.', 'type': 'invalid_request_error'}
Seed 4 (default) | Score: 0.0% | GC: 0.0% | ISS: 0.0% | ToM: 0.0%
  gold_rush: GC=0.0% ISS=0.0% ToM=0.0%
  water_crisis: GC=0.0% ISS=0.0% ToM=0.0%
  spice_wars: GC=0.0% ISS=0.0% ToM=0.0%
  hidden_gem: GC=0.0% ISS=0.0% ToM=0.0%
